# Notebook Kiểm Thử Truy Vấn Tương Tác GQA LightRAG

Notebook này cung cấp giao diện tương tác để kiểm thử tất cả **9 loại truy vấn** được hỗ trợ bởi hệ thống Đồ Thị Tri Thức GQA LightRAG.

## Các Loại Truy Vấn:
1. **Entity Search** - Tìm đối tượng theo khái niệm và thuộc tính
2. **Statistical Knowledge** - Xác suất đồng xuất hiện giữa các khái niệm
3. **Similarity Search** - Tìm đối tượng tương tự dựa trên thuộc tính
4. **Relational Path** - Tìm đường đi giữa các khái niệm
5. **Negative Constraints** - Tìm ảnh có A nhưng không có B
6. **Comparative** - So sánh số lượng/thuộc tính giữa các ngữ cảnh
7. **Hierarchical** - Tìm kiếm thực thể theo danh mục
8. **Anomaly Detection** - Tìm quan hệ hiếm/bất thường
9. **Visual-Attribute Constraint** - Tìm kiếm đa điều kiện

## Hướng Dẫn Sử Dụng:
- Chạy ô **Setup** trước để khởi tạo giao diện truy vấn
- Mỗi phần chứa một chuỗi truy vấn có thể tùy chỉnh - sửa đổi để kiểm thử các truy vấn khác nhau
- Chạy ô để xem quá trình suy luận và kết quả

## Thiết Lập: Import Thư Viện và Khởi Tạo Giao Diện Truy Vấn

Chạy ô này trước để nạp Đồ Thị Tri Thức và khởi tạo giao diện truy vấn.

### Tùy Chọn Cấu Hình:
- **DATASET_SCALE**: Chọn kích thước dataset ('1k', '10k', hoặc 'full')
- **RESULT_LIMIT**: Số lượng kết quả tối đa trả về cho mỗi truy vấn (mặc định: 50)

**Lưu ý:** Nếu gặp lỗi import, vui lòng khởi động lại kernel (Kernel → Restart) và chạy lại ô này.

In [19]:
# Setup: Import libraries and initialize query interface
import sys
import json
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Force reload modules if they were previously imported (for development)
import importlib
for module in ['src.gqa_reasoning_engine', 'src.gqa_nl_parser', 'src.gqa_query_interface']:
    if module in sys.modules:
        importlib.reload(sys.modules[module])

# Import the query interface
from src.gqa_query_interface import QueryInterface, QueryResponse

# ============================================================
# CONFIGURATION - Modify these to change behavior
# ============================================================
DATASET_SCALE = 'full'  # Options: '1k', '10k', 'full'
RESULT_LIMIT = 50       # Maximum number of results to return per query

# Initialize the query interface
print("=" * 70)
print(f"Initializing GQA LightRAG Query Interface")
print(f"  Scale: {DATASET_SCALE}")
print(f"  Result Limit: {RESULT_LIMIT}")
print("=" * 70)

interface = QueryInterface(scale=DATASET_SCALE, verbose=True)

print("\n✅ Query Interface Ready!")
print(f"   Available Query Types: 9")
print(f"   Default Result Limit: {RESULT_LIMIT}")
print("=" * 70)

Initializing GQA LightRAG Query Interface
  Scale: full
  Result Limit: 50
[INFO] Initializing Query Interface with scale: full
[INFO] Loading Knowledge Graph...
📂 Loading graph from: d:\Giáo trình 20251\IT3930E - Project III\hybrid_multimodal_retrieval\experiments\full\gqa_lightrag.gpickle
   ✅ Đã load thành công!
   📊 Nodes: 1,233,453
   🔗 Edges: 5,634,778
   🔧 Building cache...
   ✅ Cache đã sẵn sàng!
      • Instance nodes: 1,231,134
      • Concept nodes: 1,702
      • Attribute nodes: 617
      • Images: 74,289
[INFO] Query Interface ready!
       Nodes: 1,233,453
       Edges: 5,634,778

✅ Query Interface Ready!
   Available Query Types: 9
   Default Result Limit: 50


In [20]:
# Helper function to display query results in a formatted way
def display_result(response: QueryResponse, display_limit: int = 10):
    """
    Display query response in a nicely formatted way.
    
    Args:
        response: QueryResponse object from interface.query()
        display_limit: Maximum number of results to display (default: 10)
    """
    print("\n" + "=" * 70)
    print("📋 QUERY RESULTS")
    print("=" * 70)
    
    print(f"\n🔍 Query Type: {response.query_type.upper().replace('_', ' ')}")
    print(f"📝 Original Query: \"{response.original_query}\"")
    print(f"🎯 Parse Confidence: {response.parse_confidence:.2%}")
    print(f"📦 Parsed Parameters: {json.dumps(response.parsed_params, indent=2)}")
    
    if not response.success:
        print(f"\n❌ ERROR: {response.error_message}")
        return
    
    print("\n" + "-" * 70)
    print("🔬 REASONING TRACE:")
    print("-" * 70)
    for i, step in enumerate(response.reasoning_trace, 1):
        print(f"  {step}")
    
    print("\n" + "-" * 70)
    print("📊 RESULTS:")
    print("-" * 70)
    
    if response.results is None:
        print("  (No results)")
    elif isinstance(response.results, list):
        total_count = len(response.results)
        if total_count == 0:
            print("  (No matching results found)")
        else:
            print(f"  Found {total_count} results (showing up to {min(display_limit, total_count)}):")
            for i, item in enumerate(response.results[:display_limit], 1):
                if isinstance(item, dict):
                    # Compact display for dict items
                    display = {k: v for k, v in item.items() if k in [
                        'node_id', 'image_id', 'name', 'attributes', 
                        'count', 'probability', 'path', 'relation',
                        'similarity_score', 'frequency', 'subject_concept',
                        'object_concept', 'path_length'
                    ]}
                    print(f"  {i}. {display}")
                else:
                    print(f"  {i}. {item}")
            if total_count > display_limit:
                print(f"  ... and {total_count - display_limit} more results (increase display_limit to see more)")
    elif isinstance(response.results, dict):
        for key, value in response.results.items():
            if isinstance(value, dict) and 'count' in value:
                print(f"  • {key}: {value['count']} instances")
            elif isinstance(value, (int, float, str, bool)):
                print(f"  • {key}: {value}")
            elif isinstance(value, list) and len(value) <= 5:
                print(f"  • {key}: {value}")
            elif isinstance(value, list):
                print(f"  • {key}: [{len(value)} items]")
    else:
        print(f"  {response.results}")
    
    print("\n" + "=" * 70)

print("✅ Helper function 'display_result()' defined!")
print(f"   Display limit: Use display_result(response, display_limit=N) to customize")

✅ Helper function 'display_result()' defined!
   Display limit: Use display_result(response, display_limit=N) to customize


---
## 🧪 Random Query

**Kiểm thử bất kỳ truy vấn nào mà không cần biết loại truy vấn.**

Hệ thống sẽ tự động:
- Phân tích câu hỏi và xác định loại truy vấn
- Thực hiện suy luận phù hợp
- Hiển thị kết quả chi tiết

**Mẹo:** Bạn có thể tham khảo các ví dụ truy vấn ở các phần bên dưới!

In [18]:
# ============================================================
# CUSTOM QUERY TEST - Nhập truy vấn bất kỳ của bạn
# ============================================================
# ✏️ NHẬP TRUY VẤN CỦA BẠN Ở ĐÂY:
custom_query = "How often are children having a hat"
result_limit = RESULT_LIMIT  # Tùy chỉnh số lượng kết quả
# ============================================================

print(f"🔎 Testing Custom Query: \"{custom_query}\"")
print("⏳ Processing...")
response = interface.query(custom_query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Testing Custom Query: "How often are children having a hat"
⏳ Processing...

📋 QUERY RESULTS

🔍 Query Type: VISUAL ATTRIBUTE CONSTRAINT
📝 Original Query: "How often are children having a hat"
🎯 Parse Confidence: 60.00%
📦 Parsed Parameters: {
  "main_concept": "children",
  "main_attributes": [],
  "relation": "in",
  "related_concept": "hat",
  "related_attributes": []
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Tìm 'children' → 382 instances
  HOP 3: Tìm 'hat' → 7016 instances
  HOP 4: Lọc 'hat' theo [] → 7016 instances
  HOP 5: Tìm quan hệ 'in' giữa hai tập → 5 cặp thỏa mãn
  FINAL: Trả về 5 kết quả

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  Found 5 results (showing up to 5):
  1. {'relation': 'wearing'}
  2. {'relation': 'wearing'}
  3. {'rela

---
## 1. Truy Vấn Entity Search

**Mục đích:** Tìm đối tượng trong đồ thị tri thức theo tên khái niệm và/hoặc thuộc tính.

**Ví dụ các truy vấn:**
- "Find all red cars"
- "Show me white shirts"
- "Get large wooden tables"

**Sửa truy vấn bên dưới và chạy ô:**

In [47]:
# ============================================================
# QUERY TYPE 1: ENTITY SEARCH
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Find all red cars"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Entity Search Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Entity Search Query: "Find all red cars"

📋 QUERY RESULTS

🔍 Query Type: ENTITY SEARCH
📝 Original Query: "Find all red cars"
🎯 Parse Confidence: 80.00%
📦 Parsed Parameters: {
  "concept": "car",
  "attributes": [
    "red"
  ]
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Từ Global Concept 'Concept:car' → Tìm thấy 11404 instances qua cạnh 'instance_of'
  HOP 2: Từ Global Attribute 'Attr:red' → Tìm thấy 26478 instances qua cạnh 'has_attribute'
           → Giao với tập hiện tại → Còn 780 instances
  FINAL: Trả về 50 kết quả (giới hạn 50)

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  Found 50 results (showing up to 20):
  1. {'node_id': '2358675:2230332', 'image_id': '2358675', 'name': 'car', 'attributes': ['red']}
  2. {'node_id': '145:1624987'

---
## 2. Truy Vấn Statistical Knowledge

**Mục đích:** Tính xác suất/tần suất đồng xuất hiện giữa hai khái niệm.

**Ví dụ các truy vấn:**
- "How often are dogs near people"
- "What is the probability of finding shirt near man"
- "How likely is it to find window near building"

**Sửa truy vấn bên dưới và chạy ô:**

In [48]:
# ============================================================
# QUERY TYPE 2: STATISTICAL KNOWLEDGE
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "How likely is it to find window near buildings?"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Statistical Knowledge Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Statistical Knowledge Query: "How likely is it to find window near buildings?"

📋 QUERY RESULTS

🔍 Query Type: STATISTICAL KNOWLEDGE
📝 Original Query: "How likely is it to find window near buildings?"
🎯 Parse Confidence: 90.00%
📦 Parsed Parameters: {
  "concept_a": "buildings",
  "concept_b": "window"
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Global Concept 'Concept:buildings' → 1559 instances qua 'instance_of'
  HOP 2: Global Concept 'Concept:window' → 35907 instances qua 'instance_of'
  HOP 3: Duyệt 1559 instances của 'buildings' → Tìm thấy 384 cặp có quan hệ với instances của 'window'
  FINAL: P(window | gần buildings) = 384/1559 = 0.2463

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  • probability: 0.2463
  • co_occurrences: 384
  • tot

---
## 3. Truy Vấn Similarity Search

**Mục đích:** Tìm các đối tượng tương tự với mô tả tham chiếu (dựa trên thuộc tính chung).

**Ví dụ các truy vấn:**
- "Find objects similar to large green tree"
- "What looks like a red car"
- "Find things that resemble white shirt"

**Sửa truy vấn bên dưới và chạy ô:**

In [49]:
# ============================================================
# QUERY TYPE 3: SIMILARITY SEARCH
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Find objects similar to large green tree"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Similarity Search Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Similarity Search Query: "Find objects similar to large green tree"

📋 QUERY RESULTS

🔍 Query Type: SIMILARITY SEARCH
📝 Original Query: "Find objects similar to large green tree"
🎯 Parse Confidence: 90.00%
📦 Parsed Parameters: {
  "concept": "tree",
  "attributes": [
    "large",
    "green"
  ]
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Sử dụng tham số đầu vào → Concept='tree', Attributes=['large', 'green']
  HOP 2: Global Concept 'Concept:tree' → 23608 candidate instances
  HOP 3: Tính similarity dựa trên attributes chung → 518 instances có ≥2 attributes chung
  FINAL: Sắp xếp theo similarity score → Trả về top 50 kết quả

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  Found 50 results (showing up to 20):
  1. {'node_id': '2379508:4592974',

---
## 4. Truy Vấn Relational Path

**Mục đích:** Tìm đường đi/kết nối giữa hai khái niệm trong đồ thị tri thức.

**Ví dụ các truy vấn:**
- "What paths connect man to shirt"
- "How is man connected to shirt"
- "Find path from plate to table"
- "Connection between dog and person"

**Sửa truy vấn bên dưới và chạy ô:**

In [50]:
# ============================================================
# QUERY TYPE 4: RELATIONAL PATH
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Find paths from plates to tables"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Relational Path Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Relational Path Query: "Find paths from plates to tables"

📋 QUERY RESULTS

🔍 Query Type: ENTITY SEARCH
📝 Original Query: "Find paths from plates to tables"
🎯 Parse Confidence: 80.00%
📦 Parsed Parameters: {
  "concept": "table",
  "attributes": []
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Từ Global Concept 'Concept:table' → Tìm thấy 10666 instances qua cạnh 'instance_of'
  FINAL: Trả về 50 kết quả (giới hạn 50)

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  Found 50 results (showing up to 20):
  1. {'node_id': '2322056:4177904', 'image_id': '2322056', 'name': 'table', 'attributes': []}
  2. {'node_id': '2356300:3772701', 'image_id': '2356300', 'name': 'table', 'attributes': ['patterned', 'beige']}
  3. {'node_id': '2359381:793784', 'image_i

---
## 5. Truy Vấn Negative Constraints

**Mục đích:** Tìm các ảnh có chứa một khái niệm nhưng KHÔNG có khái niệm khác.

**Ví dụ các truy vấn:**
- "Find images with trees but without sky"
- "Find images with man but not woman"
- "Images containing dog but no cat"

**Sửa truy vấn bên dưới và chạy ô:**

In [51]:
# ============================================================
# QUERY TYPE 5: NEGATIVE CONSTRAINTS
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Find images with trees but without sky"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Negative Constraints Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Negative Constraints Query: "Find images with trees but without sky"

📋 QUERY RESULTS

🔍 Query Type: NEGATIVE CONSTRAINTS
📝 Original Query: "Find images with trees but without sky"
🎯 Parse Confidence: 95.00%
📦 Parsed Parameters: {
  "concept_present": "trees",
  "concept_absent": "sky"
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Concept 'trees' → 10645 instances trong 8502 images
  HOP 2: Concept 'sky' → 17980 instances trong 16585 images
  HOP 3: Images có cả 2 khái niệm: 3889 images
         Phép hiệu: 8502 (có 'trees') - 3889 (có cả 2) = 4613 images thỏa mãn
  FINAL: Trả về 50 images (giới hạn 50)

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  Found 50 results (showing up to 20):
  1. {'image_id': '2362967'}
  2. {'image_id': '2373451'}
  

---
## 6. Truy Vấn Comparative

**Mục đích:** So sánh số lượng hoặc thuộc tính giữa các ngữ cảnh hoặc khái niệm khác nhau.

**Ví dụ các truy vấn:**
- "Compare the number of people indoors vs outdoors"
- "Compare chairs in kitchen and living room"
- "Is white more common on wall or bed"

**Sửa truy vấn bên dưới và chạy ô:**

In [52]:
# ============================================================
# QUERY TYPE 6: COMPARATIVE
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Compare chairs in kitchen and living room."
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Comparative Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Comparative Query: "Compare chairs in kitchen and living room."

📋 QUERY RESULTS

🔍 Query Type: COMPARATIVE
📝 Original Query: "Compare chairs in kitchen and living room."
🎯 Parse Confidence: 95.00%
📦 Parsed Parameters: {
  "target_concept": "chairs",
  "context_a": "kitchen",
  "context_b": "living room"
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Tìm bối cảnh 'kitchen' → 959 instances trong 956 images
  HOP 2: Tìm bối cảnh 'living room' → 260 instances trong 259 images
  HOP 3: Tìm target concept 'chairs' → 700 instances tổng cộng
  HOP 4: Lọc 'chairs' theo bối cảnh → Trong 'kitchen': 28, Trong 'living room': 19
  FINAL: 'kitchen' có 28 'chairs', 'living room' có 19 'chairs' → kitchen nhiều hơn

----------------------------------------------------------------------
📊 RESULTS:
----------------------------------------------------------------------
  

---
## 7. Truy Vấn Hierarchical

**Mục đích:** Tìm tất cả các thực thể thuộc một danh mục cha (ví dụ: tất cả các loại phương tiện).

**Ví dụ các truy vấn:**
- "Show all types of vehicles"
- "List all furniture"
- "What are the electronic devices"
- "Count animals"

**Sửa truy vấn bên dưới và chạy ô:**

In [53]:
# ============================================================
# QUERY TYPE 7: HIERARCHICAL
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Show all types of vehicles"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Hierarchical Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Hierarchical Query: "Show all types of vehicles"

📋 QUERY RESULTS

🔍 Query Type: HIERARCHICAL
📝 Original Query: "Show all types of vehicles"
🎯 Parse Confidence: 90.00%
📦 Parsed Parameters: {
  "category": "vehicles",
  "child_concepts": [
    "car",
    "truck",
    "bus",
    "motorcycle",
    "bicycle",
    "bike",
    "train",
    "plane",
    "boat"
  ]
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Category 'vehicle' → Bao gồm các child concepts: ['car', 'bus', 'truck', 'motorcycle', 'bicycle', 'bike', 'train', 'airplane']
  HOP 2: Duyệt 8 child concepts → Tổng cộng 28774 instances
  HOP 3: Thống kê số lượng theo từng loại:
         • car: 11404 instances
         • bus: 3179 instances
         • train: 3163 instances
         • airplane: 2955 instances
         • truck: 2417 instances
         • bike: 2232 instances
         • motorcycle: 2100 in

---
## 8. Truy Vấn Anomaly Detection

**Mục đích:** Tìm các cặp đối tượng-quan hệ hiếm hoặc bất thường trong đồ thị tri thức.

**Ví dụ các truy vấn:**
- "What unusual object-relation pairs exist"
- "Find rare relations"
- "Top 5 rare relations"
- "Find unusual cases of dog on table"

**Sửa truy vấn bên dưới và chạy ô:**

In [54]:
# ============================================================
# QUERY TYPE 8: ANOMALY DETECTION
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "What unusual object-relation pairs exist"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Anomaly Detection Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Anomaly Detection Query: "What unusual object-relation pairs exist"

📋 QUERY RESULTS

🔍 Query Type: ANOMALY DETECTION
📝 Original Query: "What unusual object-relation pairs exist"
🎯 Parse Confidence: 95.00%
📦 Parsed Parameters: {
  "find_rare_relations": true
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Duyệt tất cả các cạnh semantic_relation trong đồ thị...
  HOP 2: Tìm thấy 475963 loại bộ ba (S-P-O) khác nhau
  HOP 3: Lọc các quan hệ có tần suất < 2 → Tìm thấy 235599 quan hệ hiếm gặp
  FINAL: Trả về 50 quan hệ hiếm gặp nhất
         1. 'straw' --[to the right of]--> 'tablecloth' (xuất hiện 1 times)
         2. 'meat' --[inside]--> 'plate' (xuất hiện 1 times)
         3. 'tablecloth' --[to the left of]--> 'straw' (xuất hiện 1 times)
         4. 'plate' --[with]--> 'meal' (xuất hiện 1 times)
         5. 'boy' --[to the left of]--> 'twigs' (xuất hiện 1

---
## 9. Truy Vấn Visual-Attribute Constraint (Tìm Kiếm Đa Thuộc Tính)

**Mục đích:** Tìm các đối tượng thỏa mãn nhiều ràng buộc thuộc tính.

**Ví dụ các truy vấn:**
- "Find large wooden brown tables"
- "Find red plastic cups"
- "Tall man wearing black shirt"

**Sửa truy vấn bên dưới và chạy ô:**

In [55]:
# ============================================================
# QUERY TYPE 9: VISUAL-ATTRIBUTE CONSTRAINT (Multi-Attribute)
# ============================================================
# ✏️ CUSTOMIZE YOUR QUERY HERE:
query = "Find large wooden brown tables"
result_limit = RESULT_LIMIT  # You can override this per query
# ============================================================

print(f"🔎 Running Visual-Attribute Constraint Query: \"{query}\"")
response = interface.query(query, limit=result_limit)
display_result(response, display_limit=20)

🔎 Running Visual-Attribute Constraint Query: "Find large wooden brown tables"

📋 QUERY RESULTS

🔍 Query Type: ENTITY SEARCH
📝 Original Query: "Find large wooden brown tables"
🎯 Parse Confidence: 80.00%
📦 Parsed Parameters: {
  "concept": "table",
  "attributes": [
    "large",
    "wooden",
    "brown"
  ]
}

----------------------------------------------------------------------
🔬 REASONING TRACE:
----------------------------------------------------------------------
  HOP 1: Từ Global Concept 'Concept:table' → Tìm thấy 10666 instances qua cạnh 'instance_of'
  HOP 2: Từ Global Attribute 'Attr:large' → Tìm thấy 19466 instances qua cạnh 'has_attribute'
           → Giao với tập hiện tại → Còn 77 instances
  HOP 3: Không tìm thấy Attribute Node 'Attr:wooden' (bỏ qua attribute này, không ảnh hưởng đến kết quả)
  HOP 4: Từ Global Attribute 'Attr:brown' → Tìm thấy 34636 instances qua cạnh 'has_attribute'
           → Giao với tập hiện tại → Còn 26 instances
  FINAL: Trả về 26 kết quả (giới h

---
## Chạy Tất Cả Các Loại Truy Vấn (Kiểm Thử Tổng Hợp)

Chạy tuần tự tất cả 9 loại truy vấn để xác minh hệ thống hoạt động đúng.

In [56]:
# ============================================================
# RUN ALL 9 QUERY TYPES - SUMMARY TEST
# ============================================================

test_queries = [
    ('Entity Search', 'Find all red cars'),
    ('Statistical', 'How often are dogs near people'),
    ('Similarity', 'Find objects similar to large green tree'),
    ('Relational Path', 'What paths connect man to shirt'),
    ('Negative Constraint', 'Find images with trees but without sky'),
    ('Comparative', 'Compare the number of people indoors vs outdoors'),
    ('Hierarchical', 'Show all types of vehicles'),
    ('Anomaly', 'What unusual object-relation pairs exist'),
    ('Multi-Attribute', 'Find large wooden brown tables'),
]

print('=' * 70)
print('COMPREHENSIVE QUERY TEST - ALL 9 TYPES')
print('=' * 70)

success_count = 0

for i, (name, q) in enumerate(test_queries, 1):
    print(f'\n[{i}/9] {name}')
    print(f'      Query: "{q}"')
    try:
        result = interface.query(q)
        if result.success:
            r = result.results
            # Extract count based on result type
            if isinstance(r, dict):
                if 'total_instances' in r:
                    count = r['total_instances']
                elif 'count_context_a' in r:
                    count = f"A:{r.get('count_context_a',0)} B:{r.get('count_context_b',0)}"
                elif 'frequency' in r or 'probability' in r:
                    count = f"{r.get('probability', r.get('frequency', 0)):.2%}" if r.get('probability', r.get('frequency', 0)) else 'N/A'
                elif 'total_results' in r:
                    count = r['total_results']
                elif 'results' in r:
                    count = len(r['results'])
                else:
                    count = len(r)
            elif isinstance(r, list):
                count = len(r)
            else:
                count = str(type(r))
            print(f'      ✅ Type: {result.query_type}, Results: {count}')
            success_count += 1
        else:
            print(f'      ❌ Error: {result.error_message}')
    except Exception as e:
        print(f'      ❌ Exception: {e}')

print('\n' + '=' * 70)
print(f'TEST COMPLETE: {success_count}/9 queries successful')
print('=' * 70)

COMPREHENSIVE QUERY TEST - ALL 9 TYPES

[1/9] Entity Search
      Query: "Find all red cars"
      ✅ Type: entity_search, Results: 10

[2/9] Statistical
      Query: "How often are dogs near people"
      ✅ Type: statistical_knowledge, Results: 0.13%

[3/9] Similarity
      Query: "Find objects similar to large green tree"
      ✅ Type: similarity_search, Results: 10

[4/9] Relational Path
      Query: "What paths connect man to shirt"
      ✅ Type: relational_path, Results: 10

[5/9] Negative Constraint
      Query: "Find images with trees but without sky"
      ✅ Type: negative_constraints, Results: 10

[6/9] Comparative
      Query: "Compare the number of people indoors vs outdoors"
      ✅ Type: comparative, Results: 6

[7/9] Hierarchical
      Query: "Show all types of vehicles"
      ✅ Type: hierarchical, Results: 8

[8/9] Anomaly
      Query: "What unusual object-relation pairs exist"
      ✅ Type: anomaly_detection, Results: 10

[9/9] Multi-Attribute
      Query: "Find large wo